# Inspect OGBench Bridge Data

Use this notebook to inspect paired cube-single bridge data: original low-dimensional ego/state observations, actions, terminals, qpos/qvel, synchronized third-person RGB frames, loader alignment, and task-specific reward relabeling.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = Path("/home/nick/infom-learning-from-obs")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from envs import ogbench_bridge_utils

# Change this to the final Unity data root after full generation completes.
DATASET_DIR = REPO_ROOT / "runs/tmp/bridge_data_smoke_20260506_002350"
RAW_FILE = "bridge-cube-single-play-v0.npz"
TASK = 1
SAMPLE_COUNT = 8
DATASET_DIR


## Raw File Summary


In [ ]:
raw_path = DATASET_DIR / RAW_FILE
with np.load(raw_path) as raw:
    raw_summary = {key: {"shape": raw[key].shape, "dtype": str(raw[key].dtype)} for key in raw.files}
raw_summary


## Visual Samples

Each panel is the third-person RGB frame saved for the same transition as the listed ego/state/action data.


In [ ]:
with np.load(raw_path) as raw:
    n = len(raw["observations"])
    sample_idxs = np.unique(np.linspace(0, n - 1, min(SAMPLE_COUNT, n), dtype=int))
    frames = raw["third_person_observations"][sample_idxs]
    observations = raw["observations"][sample_idxs]
    actions = raw["actions"][sample_idxs]
    terminals = raw["terminals"][sample_idxs]
    qpos = raw["qpos"][sample_idxs]
    episode_ids = raw["episode_ids"][sample_idxs]
    timesteps = raw["timesteps"][sample_idxs]

cols = min(4, len(sample_idxs))
rows = int(np.ceil(len(sample_idxs) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows), squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for ax, idx, frame, ob, action, terminal, qp, ep, t in zip(axes.flat, sample_idxs, frames, observations, actions, terminals, qpos, episode_ids, timesteps):
    ax.imshow(frame)
    block_xyz = qp[14:17]
    ax.set_title(
        f"idx={idx} ep={ep} t={t} terminal={bool(terminal)}\n"
        f"action={np.round(action, 2)}\nblock_xyz={np.round(block_xyz, 3)}",
        fontsize=8,
    )
plt.tight_layout()


## Transition Alignment Check

The bridge loader should pair each non-terminal current transition with the next ego/state and next third-person frame.


In [ ]:
loaded = ogbench_bridge_utils.load_dataset(str(raw_path))
with np.load(raw_path) as raw:
    raw_terminals = raw["terminals"].astype(bool)
    valid_raw_idxs = np.flatnonzero(~raw_terminals)
    check = np.unique(np.linspace(0, len(valid_raw_idxs) - 1, min(SAMPLE_COUNT, len(valid_raw_idxs)), dtype=int))
    raw_current = valid_raw_idxs[check]
    raw_next = raw_current + 1
    alignment = {
        "observations": np.allclose(loaded["observations"][check], raw["observations"][raw_current]),
        "next_observations": np.allclose(loaded["next_observations"][check], raw["observations"][raw_next]),
        "third_person_observations": np.array_equal(loaded["third_person_observations"][check], raw["third_person_observations"][raw_current]),
        "next_third_person_observations": np.array_equal(loaded["next_third_person_observations"][check], raw["third_person_observations"][raw_next]),
        "actions": np.allclose(loaded["actions"][check], raw["actions"][raw_current]),
        "terminals": np.array_equal(loaded["terminals"][check].astype(bool), raw["terminals"][raw_next].astype(bool)),
    }
alignment


## Task Reward Relabeling Summary

This shows how the same physical cube-single play files become task1-task5 datasets at load time.


In [ ]:
task_rows = []
for task in range(1, 6):
    dataset_name = f"bridge-cube-single-play-singletask-task{task}-v0"
    _, train, val = ogbench_bridge_utils.make_env_and_datasets(dataset_name, dataset_dir=str(DATASET_DIR))
    for split_name, dataset in [("train", train), ("val", val)]:
        rewards = dataset["rewards"]
        task_rows.append({
            "task": task,
            "split": split_name,
            "size": int(dataset["observations"].shape[0]),
            "reward_min": float(np.min(rewards)),
            "reward_mean": float(np.mean(rewards)),
            "reward_max": float(np.max(rewards)),
            "reward_ge_zero_frac": float(np.mean(rewards >= 0.0)),
        })
task_rows


## Inspect One Transition In Detail


In [ ]:
idx = int(sample_idxs[0])
with np.load(raw_path) as raw:
    detail = {
        "idx": idx,
        "episode_id": int(raw["episode_ids"][idx]),
        "timestep": int(raw["timesteps"][idx]),
        "terminal": bool(raw["terminals"][idx]),
        "observation_first_10": np.round(raw["observations"][idx][:10], 4).tolist(),
        "action": np.round(raw["actions"][idx], 4).tolist(),
        "qpos_shape": raw["qpos"][idx].shape,
        "qvel_shape": raw["qvel"][idx].shape,
        "third_person_shape": raw["third_person_observations"][idx].shape,
    }
detail
